In [ ]:
from pathlib import Path
import sys
import os

root = Path.cwd().resolve()
if root.name == "model" or root.name == "data_understanding":
    root = root.parents[1]
elif root.name == "notebook":
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

from srcs.evaluation.scorecard import ScorecardScaler
from srcs.evaluation.shap_analysis import generate_lgbm_shap_analysis
from srcs.pipelines.core.config_loader import load_yaml_config

print("Project Root:", root)
print("SHAP version:", shap.__version__)


d:\Anaconda\envs\credit-risk\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project Root: E:\credit-risk-modelling
SHAP version: 0.51.0


### Export validation data

In [2]:
config_path = root / "config" / "path.yaml"
paths = load_yaml_config(config_path)

valid_path = root / paths.get("data_dir", "Home Credit Dataset") / paths["dirs"]["processed_valid"] / paths["files"]["main_table"]
print(f"Loading data from: {valid_path}")

df_valid = pd.read_csv(valid_path).replace([np.inf, -np.inf], np.nan)
y_valid = df_valid["TARGET"]
X_valid = df_valid.drop(columns=["TARGET"])

print(f"Tổng số hồ sơ validation: {len(df_valid):,}")
print(f"Tỷ lệ nợ xấu thực tế (Default Rate): {y_valid.mean()*100:.2f}%")


Loading data from: E:\credit-risk-modelling\Home Credit Dataset\processed\valid\main_table.csv
Tổng số hồ sơ validation: 61,502
Tỷ lệ nợ xấu thực tế (Default Rate): 8.07%


#### Credit Score Card Model

In [4]:
# Import lr model
lr_model_path = root / "config" / "model" / "lr_model.joblib"
lr_model = joblib.load(lr_model_path)

# Initialize: Base Score = 600, Base Odds = 50:1, PDO = 20
scaler = ScorecardScaler(base_score=600, base_odds=50.0, pdo=20)
print(f"Scorecard Constants: Factor = {scaler.factor:.4f} | Offset = {scaler.offset:.4f}")

# Scorecard Points Table
scorecard_df = scaler.build_scorecard_table(lr_model)
print(f"Tổng số biến được chọn vào Scorecard: {len(scorecard_df)}")
scorecard_df.head(15)


Scorecard Constants: Factor = 28.8539 | Offset = 487.1229
Tổng số biến được chọn vào Scorecard: 200


,Feature,Coefficient (beta),Odds Ratio,Std_Dev,Points_Delta_per_Std,Risk_Direction
0,EXPECTED_INTEREST,0.8445,2.3268,7.184165e+05,-17505788.7,High Risk (Decreases Score)
1,EXPECTED_CNT_PAYMENT,-0.8264,0.4376,1.850140e+01,441.2,Low Risk (Increases Score)
2,DAYS_BIRTH,0.7943,2.2130,1.194830e+01,-273.9,High Risk (Decreases Score)
3,EXT_SOURCE_1,-0.7774,0.4596,1.808000e-01,4.1,Low Risk (Increases Score)
4,EXPECTED_INTEREST_SHARE,0.5220,1.6854,5.513000e-01,-8.3,High Risk (Decreases Score)
5,AMT_GOODS_PRICE,-0.5000,0.6065,3.693899e+05,5328959.0,Low Risk (Increases Score)
6,CREDIT_ANNUITY_RATIO,0.4239,1.5279,7.813800e+00,-95.6,High Risk (Decreases Score)
7,AMT_CREDIT,-0.4186,0.6580,4.025031e+05,4861084.6,Low Risk (Increases Score)
8,AMT_CREDIT_SUM_SUM_CREDITACTIVE_ACTIVE,0.4079,1.5036,2.270375e+06,-26718057.7,High Risk (Decreases Score)
9,EXT_SOURCE_MEAN,-0.3960,0.6730,1.299000e-01,1.5,Low Risk (Increases Score)


#### Credit Scoring

In [5]:
lr_pipe = lr_model.calibrated_classifiers_[0].estimator if hasattr(lr_model, "calibrated_classifiers_") else lr_model
expected_lr_cols = [c for c in lr_pipe.named_steps["imputer"].feature_names_in_ if c in X_valid.columns]
X_valid_lr = X_valid[expected_lr_cols]

# Probability of Default
prob_valid_lr = lr_model.predict_proba(X_valid_lr)[:, 1]

# Transform into Scorecard Points
scores_valid = scaler.prob_to_score(prob_valid_lr)

df_scored = pd.DataFrame({
    "SK_ID_CURR": df_valid["SK_ID_CURR"],
    "Actual_Target": y_valid,
    "PD_Predicted": np.round(prob_valid_lr, 4),
    "Credit_Score": scores_valid,
})

print(f"Điểm trung bình (Mean Score): {scores_valid.mean():.1f}")
print(f"Điểm thấp nhất (Min): {scores_valid.min()} | Điểm cao nhất (Max): {scores_valid.max()}")
df_scored.head(10)


Điểm trung bình (Mean Score): 591.3
Điểm thấp nhất (Min): 479 | Điểm cao nhất (Max): 952


,SK_ID_CURR,Actual_Target,PD_Predicted,Credit_Score
0,374057,0,0.0684,562
1,370024,0,0.1553,536
2,101195,0,0.0139,610
3,316728,0,0.0435,576
4,374070,0,0.0700,562
5,115137,1,0.0724,561
6,442116,0,0.0260,592
7,170386,0,0.0099,620
8,149903,0,0.0105,618
9,335740,0,0.1107,547


### Credit Rating Tiers & Cutoffs

In [6]:
rating_bands = scaler.create_rating_bands(y_valid, scores_valid)
rating_bands


,tier,Total,Bad_Count,Population_Pct,Good_Count,Default_Rate (PD),Cum_Bad_Pct,Cum_Good_Pct,KS_Separation
0,Tier E: Very High Risk (<580),18484,3520,30.05%,14964,19.04%,70.90,26.47,44.43
1,Tier D: High Risk (580-639),42326,1440,68.82%,40886,3.4%,99.90,98.78,1.12
2,Tier C: Medium Risk (640-699),691,4,1.12%,687,0.58%,99.98,100.00,0.02
3,Tier B: Low Risk (700-749),0,0,0.0%,0,NaN,99.98,100.00,0.02
4,Tier A: Prime (750+),1,1,0.0%,0,100.0%,100.00,100.00,0.00


### Distribution of credit scores between Good vs Bad Customer

In [8]:
scores_0 = scores_valid[y_valid == 0]
scores_1 = scores_valid[y_valid == 1]

plt.figure(figsize=(10, 6))
bins = np.linspace(350, 850, 50)
plt.hist(scores_0, bins=bins, density=True, alpha=0.55, color="steelblue", label=f"Non-Default (y=0, Mean={scores_0.mean():.0f})")
plt.hist(scores_1, bins=bins, density=True, alpha=0.55, color="crimson", label=f"Default (y=1, Mean={scores_1.mean():.0f})")

plt.axvline(580, color="darkred", linestyle=":", lw=1.8, label="Ngưỡng Cutoff Tier E (<580)")
plt.axvline(640, color="orange", linestyle=":", lw=1.8, label="Ngưỡng Cutoff Tier D (<640)")
plt.axvline(700, color="forestgreen", linestyle=":", lw=1.8, label="Ngưỡng Cutoff Tier B (>=700)")

plt.xlabel("Điểm Tín Dụng (Credit Score: 300 - 850)", fontsize=11)
plt.ylabel("Mật độ quần thể (Density)", fontsize=11)
plt.title(f"Phân phối Điểm Tín Dụng Scorecard (Base={int(scaler.base_score)}, PDO={int(scaler.pdo)})", fontsize=13, pad=12)
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)
plt.show()


C:\Users\Hi Windows 11 Home\AppData\Local\Temp\ipykernel_17672\4038777737.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Adverse Action Notice

In [9]:
# Tier E: score < 580 - bad debt
rejected_idx = np.where((scores_valid < 560) & (y_valid == 1))[0][0]
sample_row = X_valid_lr.iloc[rejected_idx:rejected_idx+1]
preprocessor = lr_pipe.named_steps["preprocessor"]
imputer = lr_pipe.named_steps["imputer"]
selector = lr_pipe.named_steps["selector"]
scaler_step = lr_pipe.named_steps["scaler"]
lr_clf = lr_pipe.named_steps["model"]

X_sample_imp = imputer.transform(preprocessor.transform(sample_row))
X_sample_sel = selector.transform(X_sample_imp)
X_sample_scaled = scaler_step.transform(X_sample_sel)

coefs = lr_clf.coef_[0]
sel_feat_names = imputer.feature_names_in_[selector.selected_indices_]

# Factor * (coef * standardized_value)
points_impact = - scaler.factor * (coefs * X_sample_scaled[0])

sample_explain_df = pd.DataFrame({
    "Feature": sel_feat_names,
    "Standardized_Value": np.round(X_sample_scaled[0], 3),
    "Coefficient": np.round(coefs, 4),
    "Points_Contribution": np.round(points_impact, 1),
}).sort_values(by="Points_Contribution", ascending=True).reset_index(drop=True)

print(f"=== HỒ SƠ ỨNG VIÊN #{df_valid.iloc[rejected_idx]['SK_ID_CURR']:.0f} ===")
print(f"Điểm tín dụng: {scores_valid[rejected_idx]} điểm (Xếp hạng: Tier E - Từ chối)")
print(f"Xác suất nợ xấu dự báo (PD): {prob_valid_lr[rejected_idx]*100:.2f}%")
print()
print("TOP 5 LÝ DO LỚN NHẤT KHIẾN HỒ SƠ BỊ TRỪ ĐIỂM (REJECT REASONS):")
sample_explain_df.head(5)


=== HỒ SƠ ỨNG VIÊN #240757 ===
Điểm tín dụng: 518 điểm (Xếp hạng: Tier E - Từ chối)
Xác suất nợ xấu dự báo (PD): 25.69%

TOP 5 LÝ DO LỚN NHẤT KHIẾN HỒ SƠ BỊ TRỪ ĐIỂM (REJECT REASONS):


,Feature,Standardized_Value,Coefficient,Points_Contribution
0,EXT_SOURCE_1,-1.509,-0.7774,-33.9
1,EXT_SOURCE_MEAN,-1.566,-0.3960,-17.9
2,WEIGHTED_EXT_SOURCE,-1.480,-0.3061,-13.1
3,EXPECTED_INTEREST,0.531,0.8445,-12.9
4,ORGANIZATION_TYPE,-1.754,-0.1907,-9.6


# Explainable AI with SHAP for LGBM model

In [10]:
# 1. Import lgbm
lgbm_model_path = root / "config" / "model" / "lgbm_model.joblib"
lgbm_calib = joblib.load(lgbm_model_path)
lgbm_pipe = lgbm_calib.calibrated_classifiers_[0].estimator if hasattr(lgbm_calib, "calibrated_classifiers_") else lgbm_calib

imputer_lgbm = lgbm_pipe.named_steps["imputer"]
preprocessor_lgbm = lgbm_pipe.named_steps["preprocessor"]
selector_lgbm = lgbm_pipe.named_steps["selector"]
lgbm_clf = lgbm_pipe.named_steps["model"]

# SHAP computing for random 2000 customer profiles
np.random.seed(42)
sample_indices = np.random.choice(len(X_valid), size=2000, replace=False)
X_sub = X_valid.iloc[sample_indices].copy()
expected_lgbm_cols = [c for c in imputer_lgbm.feature_names_in_ if c in X_sub.columns]
X_sub = X_sub[expected_lgbm_cols]

X_prep_sub = preprocessor_lgbm.transform(X_sub)
X_imp_sub = imputer_lgbm.transform(X_prep_sub)
X_sel_sub = selector_lgbm.transform(X_imp_sub)
lgbm_feature_names = [str(c) for c in imputer_lgbm.feature_names_in_[selector_lgbm.selected_indices_]]
X_sel_df = pd.DataFrame(X_sel_sub, columns=lgbm_feature_names)

# 4. Tree Explainer
explainer = shap.TreeExplainer(lgbm_clf)
raw_shap_values = explainer.shap_values(X_sel_df)

if isinstance(raw_shap_values, list) and len(raw_shap_values) == 2:
    shap_values = raw_shap_values[1]
elif isinstance(raw_shap_values, np.ndarray) and raw_shap_values.ndim == 3:
    shap_values = raw_shap_values[:, :, 1]
else:
    shap_values = raw_shap_values

print(f"Đã tính toán xong SHAP Matrix! Kích thước: {shap_values.shape}")


Đã tính toán xong SHAP Matrix! Kích thước: (2000, 200)


d:\Anaconda\envs\credit-risk\Lib\site-packages\shap\explainers\_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


### 1. SHAP Summary Beeswarm Plot 

In [11]:
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_sel_df,
    max_display=15,
    show=False,
    plot_type="dot",
)
plt.title("LightGBM SHAP Summary Plot (Top 15 Nhân Tố Rủi Ro)", fontsize=13, pad=15)
plt.tight_layout()
plt.show()


C:\Users\Hi Windows 11 Home\AppData\Local\Temp\ipykernel_17672\2225629526.py:2: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(
d:\Anaconda\envs\credit-risk\Lib\site-packages\shap\plots\_beeswarm.py:1150: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
C:\Users\Hi Windows 11 Home\AppData\Local\Temp\ipykernel_17672\2225629526.py:10: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
C:\Users\Hi Windows 11 Home\AppData\Local\Temp\ipykernel_17672\2225629526.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 2. SHAP Global Feature Importance Bar Chart

In [12]:
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_sel_df,
    max_display=15,
    plot_type="bar",
    show=False,
)
plt.title("LightGBM Feature Importance (Mean |SHAP| Value)", fontsize=13, pad=15)
plt.tight_layout()
plt.show()


C:\Users\Hi Windows 11 Home\AppData\Local\Temp\ipykernel_17672\599060386.py:2: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(
d:\Anaconda\envs\credit-risk\Lib\site-packages\shap\plots\_beeswarm.py:1150: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
C:\Users\Hi Windows 11 Home\AppData\Local\Temp\ipykernel_17672\599060386.py:10: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
C:\Users\Hi Windows 11 Home\AppData\Local\Temp\ipykernel_17672\599060386.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4. SHAP Waterfall

In [16]:
# Tìm hồ sơ có xác suất nợ xấu cao trong mẫu
y_sample = y_valid.iloc[sample_indices].values
high_risk_idx = np.where((shap_values.sum(axis=1) > 1.5) & (y_sample == 1))[0][0]

base_val = explainer.expected_value
if isinstance(base_val, list):
    base_val = base_val[1]

# Tạo Explanation object cho hồ sơ này
exp_high = shap.Explanation(
    values=shap_values[high_risk_idx],
    base_values=base_val,
    data=X_sel_df.iloc[high_risk_idx].values,
    feature_names=lgbm_feature_names,
)

plt.figure(figsize=(10, 6))
shap.plots.waterfall(exp_high, max_display=10, show=False)
plt.title(f"SHAP Waterfall: Hồ Sơ Rủi Ro Cao #{df_valid.iloc[sample_indices[high_risk_idx]]['SK_ID_CURR']:.0f}", fontsize=12, pad=15)
plt.tight_layout()
plt.show()


C:\Users\Hi Windows 11 Home\AppData\Local\Temp\ipykernel_17672\3084298690.py:20: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
C:\Users\Hi Windows 11 Home\AppData\Local\Temp\ipykernel_17672\3084298690.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


#### Trường hợp B: Hồ sơ an toàn được phê duyệt (Low Risk / Prime Approval)


In [ ]:
# Tìm hồ sơ an toàn có SHAP âm lớn (đẩy điểm rủi ro xuống rất thấp)
low_risk_idx = np.where((shap_values.sum(axis=1) < -2.0) & (y_sample == 0))[0][0]

exp_low = shap.Explanation(
    values=shap_values[low_risk_idx],
    base_values=base_val,
    data=X_sel_df.iloc[low_risk_idx].values,
    feature_names=lgbm_feature_names,
)

plt.figure(figsize=(10, 6))
shap.plots.waterfall(exp_low, max_display=10, show=False)
plt.title(f"SHAP Waterfall: Hồ Sơ An Toàn Phê Duyệt #{df_valid.iloc[sample_indices[low_risk_idx]]['SK_ID_CURR']:.0f}", fontsize=12, pad=15)
plt.tight_layout()
plt.show()


### 5. SHAP Dependence Plot

In [17]:
plt.figure(figsize=(9, 6))
shap.dependence_plot(
    "EXT_SOURCE_1",
    shap_values,
    X_sel_df,
    interaction_index="DAYS_BIRTH",
    show=False,
)
plt.title("SHAP Dependence Plot: EXT_SOURCE_1 vs DAYS_BIRTH", fontsize=12, pad=12)
plt.tight_layout()
plt.show()


C:\Users\Hi Windows 11 Home\AppData\Local\Temp\ipykernel_17672\704229910.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
